# Notebook 1 — Carga y parsing de JSON (EMG-EPN-612)

Este notebook está preparado para **Google Colab** y realiza:
1. Montaje de Google Drive.
2. Importación del módulo `src.data_loader`.
3. Carga y parsing estructurado de todo el dataset a un DataFrame Pandas.
4. Guardado de metadata y seriales NumPy para la siguiente etapa de Features.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

print('Entorno base listo ✅')

Entorno base listo ✅


In [2]:
# ==============================
# Montaje de Drive (Solo para Colab)
# ==============================
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    IN_COLAB = True

    # Rutas comunes por donde Colab monta carpetas sincronizadas de PC
    import glob
    possible_paths = glob.glob('/content/drive/**/emg-classification-knn-svm-ann', recursive=True)

    if len(possible_paths) > 0:
        print(f"Búsqueda automática encontró {len(possible_paths)} carpeta(s):")
        for p in possible_paths: print(f" - {p}")

        # Priorizar la carpeta de Google Drive Desktop ('Othercomputers' o 'Ordenadores')
        pc_paths = [p for p in possible_paths if 'Othercomputers' in p or 'Ordenadores' in p]
        if pc_paths:
            PROJECT_PATH = sorted(pc_paths, key=len)[0] # Tomar la más corta que sea de PC
        else:
            PROJECT_PATH = sorted(possible_paths, key=len)[0]
        print(f"\nUsando carpeta de proyecto activa: {PROJECT_PATH}")
    else:
        # Fallback manual en caso de que glob falle
        print("\nATENCIÓN: Búsqueda automática falló. Intentando rutas manuales:")
        path_pc = '/content/drive/Othercomputers/My PC/emg-classification-knn-svm-ann'
        path_mi_pc = '/content/drive/Othercomputers/Mi PC/emg-classification-knn-svm-ann'
        path_drive = '/content/drive/MyDrive/emg-classification-knn-svm-ann'
        path_ordenadores = '/content/drive/MyDrive/Ordenadores/My PC/emg-classification-knn-svm-ann'

        if os.path.exists(path_pc): PROJECT_PATH = path_pc
        elif os.path.exists(path_mi_pc): PROJECT_PATH = path_mi_pc
        elif os.path.exists(path_drive): PROJECT_PATH = path_drive
        elif os.path.exists(path_ordenadores): PROJECT_PATH = path_ordenadores
        else:
            raise FileNotFoundError("No se encontró la carpeta en Drive.")

    os.chdir(PROJECT_PATH)
    if PROJECT_PATH not in sys.path:
        sys.path.append(PROJECT_PATH)

except Exception as e:
    print(f"Excepción en Colab: {e}")
    IN_COLAB = False
    # Si estás en local
    PROJECT_PATH = os.getcwd()
    if 'notebooks' in PROJECT_PATH:
        PROJECT_PATH = os.path.abspath('..')
        os.chdir(PROJECT_PATH)
    if PROJECT_PATH not in sys.path:
        sys.path.append(PROJECT_PATH)

print(f'\nIN_COLAB = {IN_COLAB}\nDirectorio de trabajo actual: {os.getcwd()}')

Mounted at /content/drive
Búsqueda automática encontró 2 carpeta(s):
 - /content/drive/MyDrive/Tesis_HGR/emg-classification-knn-svm-ann
 - /content/drive/Othercomputers/My PC/emg-classification-knn-svm-ann

Usando carpeta de proyecto activa: /content/drive/Othercomputers/My PC/emg-classification-knn-svm-ann

IN_COLAB = True
Directorio de trabajo actual: /content/drive/Othercomputers/My PC/emg-classification-knn-svm-ann


In [3]:
# ==============================
# Importar Módulos Propios (src/)
# ==============================
import importlib

# En Python 3.12+ de Colab, `%autoreload` a veces falla por falta del módulo 'imp'.
# Usamos un bloque try-except para cargar los módulos de src con importlib de forma manual si es necesario.
try:
    %load_ext autoreload
    %autoreload 2
except Exception as e:
    print(f"Warning: Autoreload falló ({e}). Se recargará manualmente si es necesario.")

from src.config import Config
from src.data_loader import load_all_users

print(f'Ruta TRAIN_JSON_DIR esperada: {Config.TRAIN_JSON_DIR}')
print(f'Ruta TEST_JSON_DIR esperada: {Config.TEST_JSON_DIR}')

Ruta TRAIN_JSON_DIR esperada: /content/drive/Othercomputers/My PC/emg-classification-knn-svm-ann/data/trainingJSON
Ruta TEST_JSON_DIR esperada: /content/drive/Othercomputers/My PC/emg-classification-knn-svm-ann/data/testingJSON


In [4]:
# ==============================
# Carga de Datos
# ==============================
# Puedes usar max_users=3 para una prueba rápida
MAX_USUARIOS = None  # <-- MODIFICADO A 3 USUARIOS

print(f"Cargando max {MAX_USUARIOS} usuarios de training...")
df_train = load_all_users(Config.TRAIN_JSON_DIR, max_users=MAX_USUARIOS)

print(f"Cargando max {MAX_USUARIOS} usuarios de testing...")
df_test = load_all_users(Config.TEST_JSON_DIR, max_users=MAX_USUARIOS)

df_samples = pd.concat([df_train, df_test], ignore_index=True)
print(f'Total de muestras parseadas: {len(df_samples)}')
display(df_samples.head(3))

Cargando max None usuarios de training...


Parsing JSON from trainingJSON:   0%|          | 0/50 [00:00<?, ?it/s]

Cargando max None usuarios de testing...


Parsing JSON from testingJSON:   0%|          | 0/50 [00:00<?, ?it/s]

Total de muestras parseadas: 18750


,user_id,split,sample_key,gesture_name,signal_len,n_channels,signal_raw,ground_truth,ground_truth_index
0,user1,train_samples,idx_26,fist,996,8,"[[2.0, -8.0, -2.0, -1.0, 0.0, 0.0, 1.0, 0.0], ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[529, 716]"
1,user1,train_samples,idx_27,fist,996,8,"[[-2.0, -7.0, -2.0, -4.0, -1.0, 0.0, 1.0, 0.0]...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[227, 395]"
2,user1,train_samples,idx_28,fist,992,8,"[[0.0, -1.0, -5.0, -4.0, -1.0, 0.0, -1.0, 0.0]...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[609, 778]"


In [5]:
# Extraer las variables principales
X_signals_raw = df_samples['signal_raw'].to_list()  # lista de np.ndarray [T, C]
y_gesture = df_samples['gesture_name'].to_numpy()
user_ids = df_samples['user_id'].to_numpy()
splits = df_samples['split'].to_numpy()
gt_indexes = df_samples['ground_truth_index'].to_list()

print('Ejemplo X_signals_raw[0].shape (Señal entera):', X_signals_raw[0].shape)
print('Ejemplo de groundTruthIndex (Inicio y fin del gesto):', gt_indexes[0])
print('y_gesture shape:', y_gesture.shape)
print('Gestos únicos:', sorted(np.unique(y_gesture)))

Ejemplo X_signals_raw[0].shape (Señal entera): (996, 8)
Ejemplo de groundTruthIndex (Inicio y fin del gesto): [529 716]
y_gesture shape: (18750,)
Gestos únicos: ['fist', 'open', 'pinch', 'waveIn', 'waveOut']


In [6]:
# ==============================
# Guardado de Artefactos (Processed)
# ==============================
OUTPUT_DIR = Config.PROCESSED_DIR / 'notebook1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Guardamos Metadata ligera en un CSV
meta_cols = ['user_id', 'split', 'sample_key', 'gesture_name', 'signal_len', 'n_channels']
df_samples[meta_cols].to_csv(OUTPUT_DIR / 'samples_metadata.csv', index=False)

# Guardamos DataFrame completo (incluyendo el array signal_raw) en formato pickle
df_samples.to_pickle(OUTPUT_DIR / 'samples_full.pkl')

print(f'Artefactos guardados con éxito en: {OUTPUT_DIR}')
print(os.listdir(OUTPUT_DIR))

Artefactos guardados con éxito en: /content/drive/Othercomputers/My PC/emg-classification-knn-svm-ann/data/processed/notebook1
['samples_metadata.csv', 'samples_full.pkl']


In [7]:
import os
print(os.getcwd())

/content/drive/Othercomputers/My PC/emg-classification-knn-svm-ann
